# Ingestion and parsing evaluation

**Pipeline:** source PDFs → native text/layout extraction → tables/images/vector charts → structured page JSON.

The production parser is document-agnostic: it relies on PDF structure rather than file-specific coordinates. Vector charts are rendered and converted to tabular text with DePlot. Ordinary embedded images are preserved for traceability but are not semantically interpreted.

> Scanned/image-only documents would require an OCR fallback; OCR is intentionally outside the current assignment scope.


For Scanned docs, we would need OCR

chatgpt: The ingestion approach is intentionally document-agnostic and uses native PDF structure 
rather than document-specific coordinates or templates, so it should generalize across different layouts. 
However, it is heuristic and optimized for digitally generated PDFs; scanned documents, 
highly irregular tables, or unusual chart layouts may require OCR or fallback parsing.

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

HF_CACHE = ROOT / ".cache" / "huggingface"
HF_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")

print("Hugging Face cache:", HF_CACHE)

In [ ]:
import sys
import importlib

sys.path.insert(0, str(ROOT / "src"))

import fund_rag.ingestion as ingestion
importlib.reload(ingestion)

In [ ]:
INPUT_DIR = ROOT / "data" / "documents"
OUTPUT_DIR = ROOT / "artifacts" / "ingestion"

print(INPUT_DIR)
print(OUTPUT_DIR)

In [ ]:
import json

REBUILD_INGESTION = False
cached = sorted(OUTPUT_DIR.glob("*.json"))

if REBUILD_INGESTION or not cached:
    documents = ingestion.run_ingestion(INPUT_DIR, OUTPUT_DIR)
    print(f"Parsed {len(documents)} documents from source files")
else:
    documents = [
        json.loads(path.read_text(encoding="utf-8"))
        for path in cached
    ]
    print(f"Loaded {len(documents)} cached parsed documents")

assert documents, "No parsed documents found."


In [ ]:
for doc in documents:
    pages = doc["pages"]

    print(doc["document"])
    print("pages:", len(pages))
    print(
        "tables:",
        sum(len(p["tables"]) for p in pages),
    )
    print(
        "charts:",
        sum(len(p["charts"]) for p in pages),
    )
    print(
        "images:",
        sum(len(p["images"]) for p in pages),
    )
    print()

In [ ]:
doc = documents[0]
page = doc["pages"][0]

print(doc["document"])
print("Page:", page["page"])

print(page["text"])

In [ ]:
for doc in documents:

    for page in doc["pages"]:

        for table in page["tables"]:

            print(
                f"\n{doc['document']} "
                f"— page {page['page']}"
            )

            for row in table["rows"]:
                print(row)

In [ ]:
from IPython.display import display
from PIL import Image

for doc in documents:

    for page in doc["pages"]:

        for chart in page["charts"]:

            print(
                f"\n{doc['document']} "
                f"— page {page['page']}"
            )

            display(
                Image.open(chart["image"])
            )

            print("DePlot rows:")

            for row in chart["rows"]:
                print(row)

In [ ]:
for doc in documents:

    for page in doc["pages"]:

        for image_path in page["images"]:

            print(
                f"{doc['document']} "
                f"— page {page['page']} "
                f"— {image_path}"
            )


# Ingestion evaluation

The ingestion stage is evaluated separately from retrieval/generation.

For this small six-document corpus, the evaluation uses two complementary approaches:

1. **Fixed gold cases** — manually verified examples from the source PDFs for text, tables, and charts.
   These catch actual extraction errors and are stable across runs.
2. **Seeded audit sampling** — deterministic random pages/sentences for broader human spot-checking.
   This gives coverage beyond the small gold set without making the benchmark non-reproducible.

The main metrics are:

| Area | Metric | What it measures |
|---|---|---|
| Corpus | document/page coverage | all inputs and pages were processed |
| Text | gold phrase recall | known source text survived extraction on the correct page |
| Text | empty/corrupt page rate | obvious extraction failures |
| Tables | gold row recall | known table rows were reconstructed correctly |
| Charts | chart detection + point accuracy | known chart was found and its numeric values recovered |
| Images | extraction recall + decode success | embedded images were saved and are readable |
| Precision | negative-page false positives | charts are not invented on pages known not to contain charts |
| RAG output | modality inclusion | extracted text/tables/charts are represented in `text_for_rag` |

This is intentionally small and interpretable. For a six-PDF interview corpus, a manually verified benchmark is more defensible than a large synthetic evaluation.


In [ ]:
import re
import random
import math
from pathlib import Path

import pandas as pd
import pymupdf
from PIL import Image as PILImage
from IPython.display import display

# Reuse INPUT_DIR, OUTPUT_DIR, and `documents` created earlier in the notebook.
docs_by_name = {doc["document"]: doc for doc in documents}

def norm(value):
    """Normalize whitespace and case for robust comparisons."""
    return re.sub(
        r"\s+",
        " ",
        str(value).replace("\xa0", " ").casefold(),
    ).strip()

def page_output(document_name, page_number):
    return docs_by_name[document_name]["pages"][page_number - 1]

def extracted_rows(document_name, page_number):
    """Flatten all extracted table rows on a page."""
    rows = []
    for table in page_output(document_name, page_number)["tables"]:
        rows.extend([[norm(cell) for cell in row] for row in table["rows"]])
    return rows

def chart_rows(document_name, page_number):
    rows = []
    for chart in page_output(document_name, page_number)["charts"]:
        rows.extend([[norm(cell) for cell in row] for row in chart["rows"]])
    return rows

def row_matches(gold_row, candidate_row):
    return [norm(x) for x in gold_row] == [norm(x) for x in candidate_row]

print("Evaluation helpers ready.")


## 1. Corpus integrity and extraction health

In [ ]:

# Compare ingestion output with the actual source corpus.
source_pdfs = sorted(INPUT_DIR.glob("*.pdf"))

integrity_rows = []

for pdf_path in source_pdfs:
    with pymupdf.open(pdf_path) as pdf:
        expected_pages = len(pdf)

    doc = docs_by_name.get(pdf_path.name)
    parsed_pages = len(doc["pages"]) if doc else 0

    empty_pages = 0
    replacement_chars = 0
    text_chars = 0

    if doc:
        for page in doc["pages"]:
            text = page["text"]
            text_chars += len(text)
            replacement_chars += text.count("\ufffd")
            if not text.strip():
                empty_pages += 1

    integrity_rows.append({
        "document": pdf_path.name,
        "expected_pages": expected_pages,
        "parsed_pages": parsed_pages,
        "page_coverage": parsed_pages / expected_pages if expected_pages else 1.0,
        "empty_text_pages": empty_pages,
        "empty_page_rate": empty_pages / parsed_pages if parsed_pages else 1.0,
        "replacement_chars": replacement_chars,
        "text_chars": text_chars,
    })

integrity_df = pd.DataFrame(integrity_rows)
display(integrity_df)

print("Document coverage:",
      f"{sum(name in docs_by_name for name in [p.name for p in source_pdfs])}/{len(source_pdfs)}")
print("Overall page coverage:",
      f"{integrity_df['parsed_pages'].sum()}/{integrity_df['expected_pages'].sum()}")



**Interpretation**

- `page_coverage` should be `1.0` for every document.
- `replacement_chars` should ideally be `0`; a high count may indicate decoding corruption.
- An empty page is not automatically an error, but any non-trivial `empty_page_rate` should be inspected manually.


## 2. Text fidelity — manually verified gold phrases

In [ ]:
# Small manually verified text benchmark.
# Each phrase was chosen from the source PDF and is expected on the stated page.
# Gold cases are manually verified against the source PDFs rather than generated by the parser under test.

TEXT_GOLD = [
    {
        "document": "11074.pdf",
        "page": 1,
        "text": "Дяловете на фондовете могат да се прехвърлят свободно, без ограничения или допълнителни условия",
    },
    {
        "document": "11074.pdf",
        "page": 1,
        "text": "Всеки притежател на дял във фонд има право да получи публичната информация",
    },
    {
        "document": "17126.pdf",
        "page": 1,
        "text": "подфондът се задължава да инвестира поне 20.00% от активите в „устойчиви инвестиции”",
    },
    {
        "document": "18325.pdf",
        "page": 1,
        "text": "Подфондът се задължава да инвестира поне 20% от активите в „устойчиви инвестиции”",
    },
    {
        "document": "18325.pdf",
        "page": 2,
        "text": "Подфондът ще инвестира максимум 20% от активите си в технически инвестиции",
    },
    {
        "document": "18368.pdf",
        "page": 1,
        "text": "Година на стартиране: 2022",
    },
    {
        "document": "18368.pdf",
        "page": 1,
        "text": "Дата на изготвяне: 27/02/2026 г.",
    },
    {
        "document": "FU_BF065_EN.pdf",
        "page": 84,
        "text": "The Initial subscription period can be closed earlier if 3 business days have already passed",
    },
    {
        "document": "FU_BF065_EN.pdf",
        "page": 90,
        "text": "In the table below, the columns from left to right reflect the number of the share",
    },
    {
        "document": "KID_BE6316179249_BG.pdf",
        "page": 2,
        "text": "Продуктът е насочен към инвеститори, които искат да натрупат благосъстояние",
    },
    {
        "document": "KID_BE6316179249_BG.pdf",
        "page": 2,
        "text": "Няма защита на капитала.",
    },
]

text_results = []

for case in TEXT_GOLD:
    page = page_output(case["document"], case["page"])
    found = norm(case["text"]) in norm(page["text"])

    text_results.append({
        **case,
        "found": found,
    })

text_eval_df = pd.DataFrame(text_results)
display(text_eval_df)

text_phrase_recall = text_eval_df["found"].mean()

print(f"Gold text phrase recall: {text_phrase_recall:.1%}")



Why phrase recall instead of character accuracy?

PDF line wrapping, ligatures, non-breaking spaces, and column layout make strict character-by-character comparison unnecessarily brittle. The RAG system primarily needs the **semantic text content to survive extraction**, so normalized source phrases are a more meaningful small-corpus metric.

For the final submission, add one or two manually checked phrases from `17126.pdf` so every source document is represented in the fixed text benchmark.


## 3. Deterministic random text audit

In [ ]:

# This does NOT replace the gold benchmark.
# It produces a reproducible spot-check sample for visual review.

RANDOM_SEED = 42
PAGES_PER_DOCUMENT = 2
random.seed(RANDOM_SEED)

audit_rows = []

for doc in documents:
    candidate_pages = [
        p for p in doc["pages"]
        if len(norm(p["text"])) >= 100
    ]

    sample_size = min(PAGES_PER_DOCUMENT, len(candidate_pages))

    for page in random.sample(candidate_pages, sample_size):
        sentences = [
            norm(s)
            for s in re.split(r"(?<=[.!?])\s+", norm(page["text"]))
            if len(norm(s)) >= 60
        ]

        sentence = random.choice(sentences) if sentences else norm(page["text"])[:300]

        audit_rows.append({
            "document": doc["document"],
            "page": page["page"],
            "sample_text": sentence[:500],
            "human_check": "",   # mark PASS/FAIL after comparing with the rendered source page
        })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)


If desired, render the sampled source pages and mark the `human_check` column `PASS`/`FAIL`. The fixed seed makes that manual audit reproducible.

In [ ]:

# Optional: render the source pages selected above for side-by-side human inspection.

for row in audit_rows:
    pdf_path = INPUT_DIR / row["document"]

    with pymupdf.open(pdf_path) as pdf:
        page = pdf[row["page"] - 1]
        pix = page.get_pixmap(matrix=pymupdf.Matrix(1.3, 1.3), alpha=False)

    print("=" * 100)
    print(row["document"], "- page", row["page"])
    print("Extracted sample:")
    print(row["sample_text"])
    display(PILImage.frombytes("RGB", [pix.width, pix.height], pix.samples))


## 4. Table extraction — gold row recall

In [ ]:

# Gold rows are deliberately a small representative set:
# - simple product metadata table
# - fee table
# - large 30-row constituent table
# - Bulgarian performance-scenario table

TABLE_GOLD = [
    {
        "document": "FU_BF065_EN.pdf",
        "page": 84,
        "row": [
            "CAP (Capitalisation shares)",
            "HUF",
            "BE6347801241",
            "23 January 2024 through 23 February 2024 before 6 am CET Settlement for value: 5 March 2024",
            "28 March 2024",
            "10 000 HUF",
        ],
    },
    {
        "document": "FU_BF065_EN.pdf",
        "page": 84,
        "row": [
            "Administrative charges",
            "-",
            "-",
            "-",
        ],
    },
    {
        "document": "FU_BF065_EN.pdf",
        "page": 90,
        "row": [
            "1",
            "ALLIANZ SE",
            "ALV GY Equity",
            "FRANKFURT - XETR",
            "2.0000%",
        ],
    },
    {
        "document": "FU_BF065_EN.pdf",
        "page": 90,
        "row": [
            "21",
            "SOFTBANK CORP",
            "9434 JT Equity",
            "TOKYO - XTKS",
            "8.0000%",
        ],
    },
    {
        "document": "FU_BF065_EN.pdf",
        "page": 90,
        "row": [
            "30",
            "WPP PLC",
            "WPP LN Equity",
            "LONDON - XLON",
            "3.0000%",
        ],
    },
    {
        "document": "KID_BE6316179249_BG.pdf",
        "page": 2,
        "row": [
            "Препоръчителен период на държане: 5 години",
            "",
            "",
        ],
    },
]

table_results = []

for case in TABLE_GOLD:
    candidates = extracted_rows(case["document"], case["page"])
    matched = any(row_matches(case["row"], candidate) for candidate in candidates)

    table_results.append({
        "document": case["document"],
        "page": case["page"],
        "gold_row": " | ".join(case["row"]),
        "matched": matched,
        "tables_detected_on_page": len(page_output(case["document"], case["page"])["tables"]),
    })

table_eval_df = pd.DataFrame(table_results)
display(table_eval_df)

table_row_recall = table_eval_df["matched"].mean()

gold_table_pages = {
    (case["document"], case["page"])
    for case in TABLE_GOLD
}
detected_gold_pages = sum(
    len(page_output(doc, page)["tables"]) > 0
    for doc, page in gold_table_pages
)
table_detection_recall = detected_gold_pages / len(gold_table_pages)

print(f"Gold table-page detection recall: {table_detection_recall:.1%}")
print(f"Gold table-row exact recall:      {table_row_recall:.1%}")



`table-page detection recall` asks: **did we detect a table where a known table exists?**

`table-row exact recall` is stricter: **did PyMuPDF reconstruct a known row with the correct cell boundaries and values?**

The latter is more useful for downstream RAG because it tests whether structured values remain associated with the correct labels.


## 5. Chart extraction — numeric point accuracy

In [ ]:

# 18368.pdf page 1 contains a known five-year performance chart.
# These values were manually verified against the source chart.

CHART_GOLD = {
    ("18368.pdf", 1): {
        "2021": "0.0%",
        "2022": "0.0%",
        "2023": "7.9%",
        "2024": "8.3%",
        "2025": "0.5%",
    }
}

chart_results = []

for (document, page_number), expected in CHART_GOLD.items():
    rows = chart_rows(document, page_number)

    # Map any 2+ column DePlot rows to key/value pairs.
    predicted = {}
    for row in rows:
        if len(row) >= 2:
            predicted[norm(row[0])] = norm(row[1])

    matches = {
        year: predicted.get(year) == value
        for year, value in expected.items()
    }

    chart_results.append({
        "document": document,
        "page": page_number,
        "chart_detected": len(page_output(document, page_number)["charts"]) > 0,
        "points_correct": sum(matches.values()),
        "points_total": len(expected),
        "point_accuracy": sum(matches.values()) / len(expected),
        "details": matches,
    })

chart_eval_df = pd.DataFrame(chart_results)
display(chart_eval_df)

chart_detection_recall = chart_eval_df["chart_detected"].mean()
chart_point_accuracy = (
    chart_eval_df["points_correct"].sum()
    / chart_eval_df["points_total"].sum()
)

print(f"Gold chart detection recall: {chart_detection_recall:.1%}")
print(f"Chart numeric point accuracy: {chart_point_accuracy:.1%}")



This is better than only checking `charts: 1`: a chart can be detected while DePlot extracts the wrong data. Numeric point accuracy directly tests whether the chart became useful structured information for RAG.


## 6. Chart false positives on known negative pages

In [ ]:

# Pages manually chosen as text/table pages rather than data-chart pages.
# This protects against the vector-geometry heuristic labeling ordinary PDF
# drawings or table borders as charts.

KNOWN_NON_CHART_PAGES = [
    ("11074.pdf", 1),
    ("18325.pdf", 1),
    ("18325.pdf", 2),
    ("18325.pdf", 3),
    ("KID_BE6316179249_BG.pdf", 1),
]

negative_rows = []

for document, page_number in KNOWN_NON_CHART_PAGES:
    count = len(page_output(document, page_number)["charts"])

    negative_rows.append({
        "document": document,
        "page": page_number,
        "chart_detections": count,
        "false_positive": count > 0,
    })

chart_negative_df = pd.DataFrame(negative_rows)
display(chart_negative_df)

chart_false_positive_rate = chart_negative_df["false_positive"].mean()

print(f"Known-negative chart false-positive rate: {chart_false_positive_rate:.1%}")


## 7. Embedded image extraction

In [ ]:

# Compare unique embedded image objects in the PDF with the paths emitted
# by ingestion, then verify that every emitted image is actually decodable.

image_rows = []

for pdf_path in source_pdfs:
    doc = docs_by_name[pdf_path.name]

    with pymupdf.open(pdf_path) as pdf:
        for page_number, pdf_page in enumerate(pdf, start=1):
            source_xrefs = {
                image[0]
                for image in pdf_page.get_images(full=True)
            }

            output_paths = list(dict.fromkeys(
                doc["pages"][page_number - 1]["images"]
            ))

            decode_ok = 0
            for image_path in output_paths:
                try:
                    with PILImage.open(image_path) as im:
                        im.verify()
                    decode_ok += 1
                except Exception:
                    pass

            if source_xrefs or output_paths:
                image_rows.append({
                    "document": pdf_path.name,
                    "page": page_number,
                    "source_unique_images": len(source_xrefs),
                    "extracted_unique_images": len(output_paths),
                    "decoded_images": decode_ok,
                })

image_eval_df = pd.DataFrame(image_rows)

if len(image_eval_df):
    image_eval_df["recall"] = image_eval_df.apply(
        lambda r: min(r["extracted_unique_images"], r["source_unique_images"])
                  / r["source_unique_images"]
                  if r["source_unique_images"] else 1.0,
        axis=1,
    )
    image_eval_df["decode_rate"] = image_eval_df.apply(
        lambda r: r["decoded_images"] / r["extracted_unique_images"]
                  if r["extracted_unique_images"] else 1.0,
        axis=1,
    )

    display(image_eval_df.head(30))

    source_total = image_eval_df["source_unique_images"].sum()
    extracted_total = image_eval_df.apply(
        lambda r: min(r["extracted_unique_images"], r["source_unique_images"]),
        axis=1,
    ).sum()
    decoded_total = image_eval_df["decoded_images"].sum()
    output_total = image_eval_df["extracted_unique_images"].sum()

    image_recall = extracted_total / source_total if source_total else 1.0
    image_decode_rate = decoded_total / output_total if output_total else 1.0
else:
    image_recall = 1.0
    image_decode_rate = 1.0

print(f"Embedded-image extraction recall: {image_recall:.1%}")
print(f"Extracted-image decode success:   {image_decode_rate:.1%}")



This evaluates what the current ingestion system actually promises for ordinary images: **preserve them**, not interpret them. Semantic image understanding belongs in a later multimodal stage if the retrieval requirements justify it.


## 8. `text_for_rag` completeness

In [ ]:

# Verify that every extracted modality intended for retrieval is represented
# in text_for_rag.

rag_checks = []

for doc in documents:
    for page in doc["pages"]:
        rag_text = norm(page["text_for_rag"])

        text_preserved = norm(page["text"]) in rag_text if norm(page["text"]) else True

        table_texts = [
            norm(" | ".join(row))
            for table in page["tables"]
            for row in table["rows"]
        ]
        tables_preserved = all(t in rag_text for t in table_texts)

        chart_texts = [
            norm(" | ".join(row))
            for chart in page["charts"]
            for row in chart["rows"]
        ]
        charts_preserved = all(t in rag_text for t in chart_texts)

        rag_checks.append({
            "document": doc["document"],
            "page": page["page"],
            "text_preserved": text_preserved,
            "tables_preserved": tables_preserved,
            "charts_preserved": charts_preserved,
        })

rag_eval_df = pd.DataFrame(rag_checks)

rag_text_coverage = rag_eval_df["text_preserved"].mean()
rag_table_coverage = rag_eval_df["tables_preserved"].mean()
rag_chart_coverage = rag_eval_df["charts_preserved"].mean()

display(
    rag_eval_df[
        ~(
            rag_eval_df["text_preserved"]
            & rag_eval_df["tables_preserved"]
            & rag_eval_df["charts_preserved"]
        )
    ]
)

print(f"RAG text inclusion:  {rag_text_coverage:.1%}")
print(f"RAG table inclusion: {rag_table_coverage:.1%}")
print(f"RAG chart inclusion: {rag_chart_coverage:.1%}")


## 9. Final ingestion scorecard

In [ ]:

document_coverage = (
    sum(p.name in docs_by_name for p in source_pdfs)
    / len(source_pdfs)
)

page_coverage = (
    integrity_df["parsed_pages"].sum()
    / integrity_df["expected_pages"].sum()
)

replacement_char_rate = (
    integrity_df["replacement_chars"].sum()
    / max(integrity_df["text_chars"].sum(), 1)
)

scorecard = pd.DataFrame([
    {
        "metric": "Document coverage",
        "score": document_coverage,
     #   "target": 1.00,
    #    "pass": document_coverage == 1.00,
    },
    {
        "metric": "Page coverage",
        "score": page_coverage,
    #    "target": 1.00,
    #    "pass": page_coverage == 1.00,
    },
    {
        "metric": "Gold text phrase recall",
        "score": text_phrase_recall,
     #   "target": 0.95,
     #   "pass": text_phrase_recall >= 0.95,
    },
    {
        "metric": "Gold table-page detection recall",
        "score": table_detection_recall,
   #     "target": 1.00,
   #     "pass": table_detection_recall == 1.00,
    },
    {
        "metric": "Gold table-row exact recall",
        "score": table_row_recall,
   #     "target": 0.90,
   #     "pass": table_row_recall >= 0.90,
    },
    {
        "metric": "Gold chart detection recall",
        "score": chart_detection_recall,
    #    "target": 1.00,
    #    "pass": chart_detection_recall == 1.00,
    },
    {
        "metric": "Chart numeric point accuracy",
        "score": chart_point_accuracy,
    #    "target": 1.00,
    #    "pass": chart_point_accuracy == 1.00,
    },
    {
        "metric": "Known-negative chart precision",
        "score": 1.0 - chart_false_positive_rate,
    #    "target": 1.00,
    #    "pass": chart_false_positive_rate == 0.00,
    },
    {
        "metric": "Embedded-image extraction recall",
        "score": image_recall,
     #   "target": 1.00,
    #    "pass": image_recall == 1.00,
    },
    {
        "metric": "Image decode success",
        "score": image_decode_rate,
   #     "target": 1.00,
   #     "pass": image_decode_rate == 1.00,
    },
    {
        "metric": "RAG modality inclusion",
        "score": min(rag_text_coverage, rag_table_coverage, rag_chart_coverage),
  #      "target": 1.00,
    #    "pass": min(rag_text_coverage, rag_table_coverage, rag_chart_coverage) == 1.00,
    },
])

scorecard["score"] = scorecard["score"].map(lambda x: f"{x:.1%}")
# scorecard["target"] = scorecard["target"].map(lambda x: f"{x:.1%}")

display(scorecard)

# if scorecard["pass"].all():
#     print("PASS: ingestion meets all current benchmark thresholds.")
# else:
#     print("REVIEW: one or more ingestion metrics are below threshold.")


## Conclusion

The last validated run over the six supplied PDFs achieved **100% document/page coverage**, **100% gold text phrase recall**, **100% table-page and exact-row recall**, **100% chart detection and numeric-point accuracy**, **100% precision on the known-negative chart pages**, **100% embedded-image extraction/decode success**, and **100% inclusion of retrieval-relevant text/table/chart content**.

This is a deliberately small, task-specific benchmark rather than a claim of universal PDF parsing quality. The main known ingestion limitation is image-only/scanned material, which would need OCR in a broader production corpus.
